In [31]:
import numpy as np
from scipy.linalg import eig
from pathlib import Path
from copy import deepcopy as copy

from TEST.geometry import Slab
from TEST.material import Material

e3 = np.array([2.00E+01,                     1.00E-01,                               6.25E-07, 1.00E-11]) # MeV

In [32]:
mat = "LFR_3G_A"
P1consistent = False # True # 
pwd = Path.cwd()
dataMG_ref = Material(uniName=mat, energygrid=e3, datapath=str(pwd.joinpath(mat)), P1consistent=P1consistent)
G = dataMG_ref.nE

Now the direct and the adjoint eigenvalue problems will be solved in a homogeneous, cylindrical medium featured by a buckling $B^2=B_z^2+B_r^2=\displaystyle{\biggl(\frac{\pi}{H}\biggr)^2+\biggl(\frac{j_0}{R}\biggr)^2}$.

In [33]:
H = 152 # cm
B = np.pi/H
print(f"Buckling with H={H} cm is {B**2:1.3e} cm^-2")

Buckling with H=152 cm is 4.272e-04 cm^-2


In [34]:
# operator definition
L_ref = np.zeros((G, G))
Linf_ref = np.zeros((G, G))
F_ref = np.zeros((G, G))
S_ref = np.zeros((G, G))

for g in range(G):
    L_ref[g, g] = dataMG_ref.Sigma_rem[g] + dataMG_ref.Diffcoef[g] * B**2
    Linf_ref[g, g] = dataMG_ref.Sigma_rem[g]
    for h in range(G):
        F_ref[g, h] = dataMG_ref.chi_tot[g]*dataMG_ref.nu_fiss[h]*dataMG_ref.Sigma_fiss[h]
        if h != g: # this term is included in the removal XS
            S_ref[g, h] = dataMG_ref.S0[g, h]

Direct problem:
* $(\hat L - \hat S) \vec{\phi} = \displaystyle\frac{1}{k} \hat F \vec{\phi} $

Adjoint problem:
* $(\hat L^T - \hat S^T) \vec{\phi^+} = \displaystyle\frac{1}{k^+} \hat F^T \vec{\phi^+} $

where $k=k^+$ and $\braket{\hat F^T \vec{\phi^T} | \vec{\phi}} = \braket{\vec{\phi^T} | \hat F\vec{\phi}}$.

In [35]:
evlD, evcD = eig(L_ref-S_ref, b=F_ref)
kD = 1/evlD[0].real
flxD = evcD[:, 0] if evcD[:, 0].max() > 0 else -evcD[:, 0]

evlA, evcA = eig(L_ref.T-S_ref.T, b=F_ref.T)
kA = 1/evlA[0].real
flxA = evcA[:, 0] if evcA[:, 0].max() > 0 else -evcA[:, 0]

# sanity check on eigenvalues
assert 1E5*(abs(kA-kD)) < 1E-3
# sanity check on eigenvectors
assert abs(np.dot(flxA, F_ref.dot(flxD))-np.dot(F_ref.T.dot(flxA), flxD)) < 1E-12

# impose criticality
k = kD
F_ref = F_ref/k
evlD, evcD = eig(L_ref-S_ref, b=F_ref)
kD_ref = 1/evlD[0].real
flxD_ref = evcD[:, 0] if evcD[:, 0].max() > 0 else -evcD[:, 0]
flxD_ref /= F_ref.dot(flxD_ref).sum() # flxD_ref[0]

evlA, evcA = eig(L_ref.T-S_ref.T, b=F_ref.T)
kA_re = 1/evlA[0].real
flxA_ref = evcA[:, 0] if evcA[:, 0].max() > 0 else -evcA[:, 0]
flxA_ref /= flxA_ref[0]

# k infinity
evlinf, flx_inf = eig(Linf_ref-S_ref, b=F_ref)

kinf_ref = 1/evlinf[0].real

# sanity check on eigenvalues
assert 1E5*(abs(kA-kD)) < 1E-3
# sanity check on eigenvectors
assert abs(np.dot(flxA_ref, F_ref.dot(flxD_ref))-np.dot(F_ref.T.dot(flxA_ref), flxD_ref)) < 1E-12
print(f"Effective mult. param. with {G}G = {k:.10f}")

Effective mult. param. with 3G = 1.3010560560


In [36]:
Lambda_eff_ref = np.dot(flxA_ref, dataMG_ref.inv_vel*(flxD_ref)) / np.dot(flxA_ref, F_ref.dot(flxD_ref))
print(f"Total importance amount: {np.dot(flxA_ref, dataMG_ref.inv_vel*(flxD_ref))}")
print(f"Infinite mult. param. with {G}G = {kinf_ref:.6f}")
print(f"Effective mult. param. with {G}G = {kD_ref:.6f}")
print(f"Effective lifetime with {G}G = {Lambda_eff_ref*1E6:.4f} micro")

Total importance amount: 6.284632321199802e-07
Infinite mult. param. with 3G = 1.097969
Effective mult. param. with 3G = 1.000000
Effective lifetime with 3G = 0.6285 micro


In [37]:
chi = dataMG_ref.chi_tot
nsf = dataMG_ref.nuSigma_fiss/k
sc = dataMG_ref.S0.T
D = dataMG_ref.Diffcoef
r = dataMG_ref.Sigma_rem
v = 1/dataMG_ref.inv_vel
P1 = 1/(D[0]*B**2+r[0])
P2 = 1/(D[1]*B**2+r[1])
P3 = 1/(D[2]*B**2+r[2])
# analytical 3G formula for keff (chi[0]=1)
# A = (nsf[0]/(P2*P3)-nsf[0]*s[1,2]*s[2,1]+nsf[1]*s[0,1]/P3+nsf[1]*s[0,2]*s[2,1]+nsf[2]*s[0,1]*s[1,2]+nsf[2]*s[0,2]/P2)
# B = 1/(P1*P2*P3)-s[1,2]*s[2,1]/P1
# keff_3G = chi[0]*A/B
# A = (nsf[0]/(P2*P3)-nsf[0]*s[1,2]*s[2,1]+nsf[1]*s[0,1]/P3+nsf[1]*s[0,2]*s[2,1]+nsf[2]*s[0,1]*s[1,2]+nsf[2]*s[0,2]/P2)
# B = 1/(P1*P2*P3)-s[1,2]*s[2,1]/P1
s = (1-P2*P3*sc[1,2]*sc[2,1])
rs1 = dataMG_ref.Sigma_abs[1] + sc[1,2]*(1-sc[2,1]*P3)
rs2 = dataMG_ref.Sigma_abs[2] + sc[2,1]*(1-sc[1,2]*P2)
C1 = (P1*sc[0,1]+P1*sc[0,2]*P3*sc[2,1])/s
C2 = (P1*sc[0,2]+P1*sc[0,1]*P2*sc[1,2])/s
k1 = nsf[0]*P1
k2 = nsf[1]*P2*C1
k3 = nsf[2]*P3*C2
keff_3G = k1+k2+k3
print(f"keff analyt. = {keff_3G:.10f}")

keff analyt. = 1.0000000000


In [40]:
flxA_ref /= flxA_ref[0]
flxD_ref /= flxD_ref[0]

# phi3d = 1/s*(P3*sc[0,2]+P2*P3*sc[0,1]*sc[1,2])
# phi3a = 1/s*(P3*nsf[2]+P2*P3*sc[2,1]*nsf[1])
# phiD = np.array([1, sc[0,1]*P2+sc[2,1]*P2*phi3d, phi3d])
# phiA = np.array([1, P2*(nsf[1]+sc[1,2]*phi3a), phi3a])
phi1d = 1
phi2d = sc[0,1]*P2+sc[2,1]*P2*(sc[0,2]/(D[2]*B**2+rs2)+sc[0,1]*sc[1,2]/(D[1]*B**2+r[1])/(D[2]*B**2+rs2))
phi3d = sc[0,2]/(D[2]*B**2+rs2)+P2*sc[0,1]*sc[1,2]/(D[2]*B**2+rs2)

phi1a = 1
phi2a = P2*nsf[1]+sc[1,2]*P2*(nsf[2]/(D[2]*B**2+rs2)+sc[2,1]*nsf[1]*P2/(D[2]*B**2+rs2))
phi3a = nsf[2]/(D[2]*B**2+rs2)+P2*sc[2,1]*nsf[1]/(D[2]*B**2+rs2)

phiD = np.array([phi1d, phi2d, phi3d])
phiA = np.array([phi1a, phi2a, phi3a])

den_ana = phiA[0]*(nsf[0]*phiD[0]+nsf[1]*phiD[1]+nsf[2]*phiD[2])
den_num = np.dot(flxA_ref, F_ref.dot(flxD_ref))
print(f"Effective lifetime: analyt. = {den_ana:.4e}, numerical = {den_num:.4e}")

num_ana = phiA[0]*1/v[0]*phiD[0]+phiA[1]*1/v[1]*phiD[1]+phiA[2]*1/v[2]*phiD[2]
num_num = np.dot(flxA_ref, dataMG_ref.inv_vel*(flxD_ref))
print(f"Effective lifetime: analyt. = {num_ana:.4e}, numerical = {num_num:.4e}")


Effective lifetime: analyt. = 7.8358e-03, numerical = 7.8358e-03
Effective lifetime: analyt. = 4.9245e-09, numerical = 4.9245e-09


# Analytical results

In [41]:
# # C1 = (P1*sc[0,1]+P1*sc[0,2]*P3*sc[2,1])/s
# # C2 = (P1*sc[0,2]+P1*sc[0,1]*P2*sc[1,2])/s
# # k1 = nsf[0]*P1
# # k2 = nsf[1]*P2*C1
# # k3 = nsf[2]*P3*C2

# q2 = 1/s*(k2+k3-P1*P3*sc[0,2]*nsf[2])
# q3 = (k2+k3)-P1*sc[0,1]*P2*nsf[1]
# leff_3G = P1/v[0]+P2/v[1]*q2+P3/v[2]/s*q3
# print(f"leff semi-analyt. = {num_ana/den_ana*1E6:.4f} micros")
# print(f"leff analyt. = {leff_3G*1E6:.4f} micros")
# print(f"Effective lifetime with {G}G = {Lambda_eff_ref*1E6:.4f} micros")

In [42]:
lambda1 = 1/v[0]*P1
lambda2 = 1/(v[1]*(D[1]*B**2+rs1))
lambda3 = 1/(v[2]*(D[2]*B**2+rs2))

w2 = nsf[1]/(D[1]*B**2+rs1)*(sc[0,1]*P1+sc[0,2]*sc[2,1]*P1*P3) + \
     + nsf[2]/(D[2]*B**2+rs2)*(sc[0,1]*sc[1,2]*P1*P2) + \
     + nsf[2]*sc[0,2]*P1*(1/(D[2]*B**2+rs2)-1/(D[2]*B**2+r[2]))
w3 = sc[0,1]*nsf[1]*P1*(1/(D[1]*B**2+rs1)-1/(D[1]*B**2+r[1])) + \
     nsf[1]/(D[1]*B**2+rs1)*(sc[0,2]*sc[2,1]*P1*P3) + \
     nsf[2]/(D[2]*B**2+rs2)*(sc[0,2]*P1+sc[0,1]*sc[1,2]*P1*P2)

print(f"l1: {lambda1*1E6:.4f}, l2: {lambda2*1E6:.4f} w2: {w2:.3f} , l3: {lambda3*1E6:.4f} w3: {w3:.3f}")

# q2 = 1/s*(k2+k3-P1*P3*sc[0,2]*nsf[2])
# q3 = (k2+k3)-P1*sc[0,1]*P2*nsf[1]
leff_3G = lambda1+lambda2*w2+lambda3*w3 # P1/v[0]+P2/v[1]*q2+P3/v[2]/s*q3
print(f"leff semi-analyt. = {num_ana/den_ana*1E6:.4f} micros")
print(f"leff analyt. = {leff_3G*1E6:.4f} micros")
print(f"Effective lifetime with {G}G = {Lambda_eff_ref*1E6:.4f} micros")

l1: 0.1578, l2: 1.2623 w2: 0.372 , l3: 35.7224 w3: 0.000
leff semi-analyt. = 0.6285 micros
leff analyt. = 0.6285 micros
Effective lifetime with 3G = 0.6285 micros


# Analytical results collapsing from 3G to 2G

In [43]:
# # e8 = np.array([2.00E+01, 1.00E+00, 1.00E-01, 1.00E-02, 1.00E-03, 1.00E-04, 1.00E-05, 6.25E-07, 1.00E-11]) # MeV
# # e7 = np.array([2.00E+01, 1.00E+00,           1.00E-02, 1.00E-03, 1.00E-04, 1.00E-05, 6.25E-07, 1.00E-11]) # MeV
# # e6 = np.array([2.00E+01, 1.00E+00,           1.00E-02, 1.00E-03, 1.00E-04,           6.25E-07, 1.00E-11]) # MeV
# # e5 = np.array([2.00E+01, 1.00E+00,           1.00E-02,           1.00E-04,           6.25E-07, 1.00E-11]) # MeV
# # e4 = np.array([2.00E+01,                     1.00E-02,           1.00E-04,           6.25E-07, 1.00E-11]) # MeV
# # e3 = np.array([2.00E+01,                     1.00E-02,                               6.25E-07, 1.00E-11]) # MeV
e2 = np.array([2.00E+01,                                                             6.25E-07, 1.00E-11]) # MeV
# e1 = np.array([2.00E+01,                                                                       1.00E-11]) # MeV

In [44]:
data2G = copy(dataMG_ref)
data2G.nu_fiss /= k
data2G.collapse(e2, spectrum=phiD, fixdata=True,)

nG = data2G.nE
# --- operator definition
L2G = np.zeros((nG, nG))
F2G = np.zeros((nG, nG))
S2G = np.zeros((nG, nG))
Linf2G = np.zeros((nG, nG))
for g in range(nG):
    L2G[g, g] = data2G.Sigma_rem[g] + data2G.Diffcoef[g] * B**2
    Linf2G[g, g] = data2G.Sigma_abs[g]+data2G.S0[:, g].sum()-data2G.S0[g, g]
    for h in range(nG):
        F2G[g, h] = data2G.chi_tot[g]*data2G.nu_fiss[h]*data2G.Sigma_fiss[h]
        if h != g: # this term is included in the removal XS
            S2G[g, h] = data2G.S0[g, h]
# --- eigenvalue calculation
evlD, evcD = eig(L2G-S2G, b=F2G)
kD = 1/evlD[0].real
flxD2G = evcD[:, 0] if evcD[:, 0].max() > 0 else -evcD[:, 0]
flxD2G /= F2G.dot(flxD2G).sum()

evlA, evcA = eig(L2G.T-S2G.T, b=F2G.T)
kA = 1/evlA[0].real
flxA2G = evcA[:, 0] if evcA[:, 0].max() > 0 else -evcA[:, 0]
flxA2G /= F2G.dot(flxA2G).sum()

# sanity check on eigenvalues
assert 1E5*(abs(kA-kD)) < 1E-3
# sanity check on flux
# assert np.allclose(flx_few, flxD)
# sanity check on eigenvectors bi-orthogonality
assert abs(np.dot(flxA2G, F2G.dot(flxD2G))-np.dot(F2G.T.dot(flxA2G), flxD2G)) < 1E-12

Lambda_eff_2G = np.dot(flxA2G, data2G.inv_vel*(flxD2G)) / np.dot(flxA2G, F2G.dot(flxD2G))


In [45]:
chi_2G = data2G.chi_tot
nsf_2G = data2G.nuSigma_fiss
sc_2G = data2G.S0.T
D_2G = data2G.Diffcoef
r_2G = data2G.Sigma_rem
v_2G = 1/data2G.inv_vel
P1_2G = 1/(D_2G[0]*B**2+r_2G[0])
P2_2G = 1/(D_2G[1]*B**2+r_2G[1])

rs1 = data2G.Sigma_abs[0] + sc_2G[0,1]*(1-sc_2G[1,0]*P2_2G)
rs2 = data2G.Sigma_abs[1] + sc_2G[1,0]*(1-sc_2G[0,1]*P1_2G)

keff_2G = nsf_2G[0]/(D_2G[0]*B**2+rs1)+sc_2G[0,1]*nsf_2G[1]/(D_2G[0]*B**2+rs1)*P2_2G
print(f"keff analyt. = {keff_2G:.10f}")
print(f"keff numerical = {kD:.10f}")


keff analyt. = 0.9992472403
keff numerical = 0.9992472403


In [46]:
w2_2G = sc[0,1]*P1
w3_2G = sc[0,1]*P1*sc[1,2]*(sc[2,1]+nsf[2]/keff_2G)*P2*P3
lambda_3G_2G = lambda1+lambda2*w2_2G+lambda3*w3_2G

print(f"numerical 2G: {Lambda_eff_2G*1E6:.6f}")
print(f"analytical 2G: {lambda_3G_2G*1E6:.6f}")

numerical 2G: 0.834617
analytical 2G: 0.834617


In [47]:
e1 = np.array([2.00E+01,                                                                       1.00E-11]) # MeV

data1G = copy(dataMG_ref)
data1G.nu_fiss /= k
data1G.collapse(e1, spectrum=phiD, fixdata=True,)

nG = data1G.nE
# --- operator definition
L1G = np.zeros((nG, nG))
F1G = np.zeros((nG, nG))
S1G = np.zeros((nG, nG))
Linf1G = np.zeros((nG, nG))
for g in range(nG):
    L1G[g, g] = data1G.Sigma_rem[g] + data1G.Diffcoef[g] * B**2
    Linf1G[g, g] = data1G.Sigma_abs[g]+data1G.S0[:, g].sum()-data1G.S0[g, g]
    for h in range(nG):
        F1G[g, h] = data1G.chi_tot[g]*data1G.nu_fiss[h]*data1G.Sigma_fiss[h]
        if h != g: # this term is included in the removal XS
            S1G[g, h] = data1G.S0[g, h]
# --- eigenvalue calculation
evlD, evcD = eig(L1G-S1G, b=F1G)
kD = 1/evlD[0].real
flxD1G = evcD[:, 0] if evcD[:, 0].max() > 0 else -evcD[:, 0]
flxD1G /= F1G.dot(flxD1G).sum()

evlA, evcA = eig(L1G.T-S1G.T, b=F1G.T)
kA = 1/evlA[0].real
flxA1G = evcA[:, 0] if evcA[:, 0].max() > 0 else -evcA[:, 0]
flxA1G /= F1G.dot(flxA1G).sum()

# sanity check on eigenvalues
assert 1E5*(abs(kA-kD)) < 1E-3
# sanity check on flux
# assert np.allclose(flx_few, flxD)
# sanity check on eigenvectors bi-orthogonality
assert abs(np.dot(flxA1G, F1G.dot(flxD1G))-np.dot(F1G.T.dot(flxA1G), flxD1G)) < 1E-12

Lambda_eff_1G = np.dot(flxA1G, data1G.inv_vel*(flxD1G)) / np.dot(flxA1G, F1G.dot(flxD1G))
print(f"numerical 1G: {Lambda_eff_1G*1E6:.6f}")


numerical 1G: 0.836105
